# 15 — Iterative Self-Improvement Loop
The model teaches itself ARO in rounds:

```
Round 0  →  base Qwen3-Coder-30B-A3B generates data
            self-repair fixes failures with aro check
            fine-tune on passing samples
            fuse adapter → model_round_1

Round 1  →  model_round_1 generates NEW data (better quality)
            self-repair loop
            fine-tune on round_0 + round_1 data combined
            fuse → model_round_2

Round N  →  ...
```

Each round the model gets better at ARO → generates higher-quality data →
next fine-tune starts from a stronger point.

The initial dataset from notebooks 06–10 (if present) is included in round 0's training set,
so the loop builds directly on top of the warm-start and validated pairs.

**100% local, 100% open-source, runs on Apple Silicon.**

**Prerequisites:** notebooks 01–13 (or at minimum 01–04 + 06–10 + 11)
**Install:** `pip install mlx-lm`

In [1]:
import subprocess
subprocess.run(['pip', 'install', '-q', 'mlx-lm'], check=False)

CompletedProcess(args=['pip', 'install', '-q', 'mlx-lm'], returncode=0)

In [ ]:
import json
import os
import re
import random
import subprocess
import sys
import importlib
import tempfile
import time
from collections import Counter
from pathlib import Path
from mlx_lm import load, generate as mlx_generate
from mlx_lm.sample_utils import make_sampler

try:
    SCRIPT_DIR = Path(__file__).parent.resolve()
except NameError:
    SCRIPT_DIR = Path('.').resolve()

if str(SCRIPT_DIR) not in sys.path:
    sys.path.insert(0, str(SCRIPT_DIR))
import config; importlib.reload(config); from config import *

# Route all tempfile operations to the external volume — /var/folders fills up quickly
_TMP_ROOT = DATA_ROOT / 'tmp'
_TMP_ROOT.mkdir(parents=True, exist_ok=True)
os.environ['TMPDIR'] = str(_TMP_ROOT)
tempfile.tempdir = str(_TMP_ROOT)   # also set Python-level override
print(f'Temp dir: {tempfile.gettempdir()}')

ITERATIVE_MODELS_DIR.mkdir(parents=True, exist_ok=True)
ROUNDS_DIR   = DATA_ROOT  / 'rounds'
ROUNDS_DIR.mkdir(parents=True, exist_ok=True)

print(f'Data root:  {DATA_ROOT}')
print(f'Models dir: {ITERATIVE_MODELS_DIR}')
print(f'Rounds dir: {ROUNDS_DIR}')

## Loop config

In [ ]:
# Use MODEL_ID from config.py (respects TRAIN_ON_BASE flag).
# After each round this is replaced with the fused model path.
BASE_MODEL = MODEL_ID

# If NB15 produced a fused model, start from that instead of the untrained base.
# This avoids wasting round 0 re-learning what NB15 already trained.
_nb15_fused = FINETUNE_MODELS_DIR / 'round_0' / 'fused'
if (_nb15_fused / 'config.json').exists():
    BASE_MODEL = str(_nb15_fused)
    print(f'Starting from NB15 fused model: {_nb15_fused}')
else:
    print(f'No NB15 fused model found at {_nb15_fused}, starting from: {BASE_MODEL}')

NUM_ROUNDS          = 3      # how many improvement rounds to run
SAMPLES_PER_ROUND   = 300    # new samples to generate each round
MAX_REPAIR_ATTEMPTS = 3      # self-repair retries per sample
ITERS_PER_ROUND     = 400    # LoRA training iterations per round
BATCH_SIZE          = 4
LORA_RANK           = 16     # match NB15
LORA_LAYERS         = 8      # match NB15
LEARNING_RATE       = 2e-5   # match NB15
MAX_SEQ_LEN         = 4096   # match NB15

# Training stability guards (applied in run_training)
LOSS_EXPLODE_THRESHOLD = 8.0  # kill training if train_loss spikes above this
EARLY_STOP_PATIENCE    = 3    # stop if val_loss increases this many consecutive checks

# Domains to draw from during generation
DOMAINS = [
    # ── Web APIs & CRUD ──────────────────────────────────────────────────────
    'todo list API with create, list, complete, and delete',
    'product catalog API with search and pagination',
    'blog posts API with comments and draft/publish workflow',
    'task manager API with priorities and due dates',
    'event registration API with capacity limits',
    'customer orders API with status state machine',
    'employee directory API with department filtering',
    'support ticket API with severity levels',
    'appointment scheduling API with conflict detection',
    'notification preferences API per user',
    'poll and voting API with results',
    'book library API with borrowing and returns',
    'user profiles API with avatar upload',
    'subscription billing API with renewal alerts',
    'project milestones API and task tracking',
    'contact list API with tags and search',
    'feature flag manager API with rollout percentage',
    'audit log viewer API with filtering',
    'API key management with rate limits',
    'webhook delivery API with retry logic',
    'document versioning API with diff',
    'team management API with roles',
    'coupon codes API with expiry and usage limits',
    'recipe collection API with ingredient search',
    'inventory tracking API with low-stock email alerts',

    # ── Real-Time & WebSocket ────────────────────────────────────────────────
    'real-time message board with WebSocket broadcast and in-memory storage',
    'live chat application with WebSocket rooms and user nicknames',
    'collaborative text editor with WebSocket sync',
    'real-time dashboard that pushes system metrics to connected clients',
    'stock ticker that streams price updates over WebSocket',
    'notification hub that broadcasts domain events to WebSocket subscribers',
    'live auction platform with real-time bidding over WebSocket',

    # ── File & Data Operations ───────────────────────────────────────────────
    'CSV file reader that parses rows and computes column statistics',
    'log file tail watcher that monitors a directory for new .log files',
    'JSON config merger that reads multiple config files and deep-merges them',
    'file deduplicator that hashes files in a directory and reports duplicates',
    'backup rotator that copies files to a dated archive folder and prunes old backups',
    'YAML validator that reads .yaml files from a directory and reports parse errors',
    'markdown link checker that reads .md files and validates internal links exist',
    'CSV to JSON converter that reads a CSV and writes each row as JSON',
    'file metadata reporter that lists files with size, modified date, and permissions',
    'directory tree printer that recursively lists a folder structure with indentation',
    'format-aware file reader that auto-detects JSON, YAML, and CSV formats',

    # ── Data Engineering & Pipelines ─────────────────────────────────────────
    'data pipeline that reads JSON records, filters by field value, and writes results',
    'ETL job that extracts data from an API, transforms fields, and stores in repository',
    'aggregation service that groups records by category and computes sum and average',
    'map-reduce pipeline that splits a dataset, processes chunks, and merges results',
    'data quality checker that validates records against a schema and emits error events',
    'time series aggregator that buckets events by hour and computes counts per bucket',
    'deduplication pipeline that reads records, detects duplicates by key, and keeps latest',
    'data enrichment service that fetches external API data and merges into local records',
    'batch processor that reads a file line by line, transforms each, and writes output',
    'streaming word counter that reads text files and computes word frequency by count',
    'stream processor for large files with O(1) memory and running statistics',
    'set operations pipeline with union, intersect, and difference on collections',

    # ── DevOps & Deployment ──────────────────────────────────────────────────
    'deployment status tracker with rollback state machine',
    'health check monitor that polls endpoints and emits alerts on failure',
    'service registry that tracks running services with heartbeat expiry',
    'release notes generator that reads git commit messages and formats a changelog',
    'environment config manager that reads env-specific YAML and validates required keys',
    'container restart watcher that monitors a socket for crash events and logs restarts',
    'canary deployment controller with traffic percentage and automatic rollback',
    'secret rotation scheduler that tracks expiry dates and emits renewal events',
    'incident tracker with severity levels and escalation state machine',
    'uptime dashboard that stores check results and computes availability percentage',

    # ── File System Watching & Automation ────────────────────────────────────
    'file watcher that monitors a directory and emits events on create, modify, and delete',
    'hot-reload config service that watches a YAML file and re-applies settings on change',
    'image thumbnail generator that watches an upload folder and creates resized copies',
    'log rotator that watches log file size and archives when threshold is exceeded',
    'sync service that watches a source directory and mirrors changes to a destination',
    'file classifier that watches an inbox folder and moves files to subfolders by extension',
    'template renderer that watches .mustache files and re-renders HTML on change',

    # ── CLI Tools & Utilities ────────────────────────────────────────────────
    'command-line calculator that parses arithmetic expressions from arguments',
    'password generator that produces random strings with configurable length and charset',
    'JSON pretty printer that reads stdin and writes formatted JSON',
    'base64 encoder/decoder CLI that converts files or stdin',
    'UUID generator that outputs UUIDs in different formats',
    'string diff tool that compares two text files and highlights differences',
    'CSV column extractor that selects specific columns by name',
    'hash checker that computes SHA256 of files and verifies against a manifest',
    'timestamp converter that translates between Unix epoch and human-readable dates',

    # ── Monitoring & Metrics ─────────────────────────────────────────────────
    'Prometheus metrics exporter that tracks request count, latency, and error rate',
    'disk usage monitor that checks free space and emits alerts above threshold',
    'process watchdog that monitors a PID and restarts the process on crash',
    'SLA tracker that computes uptime percentage from health check results',
    'request rate limiter that throttles API calls per client with sliding window',
    'system load monitor that parses uptime and serves metrics via JSON HTTP API',

    # ── Event-Driven & Messaging ─────────────────────────────────────────────
    'order fulfillment pipeline with state transitions from placed to shipped to delivered',
    'email notification hub that listens for domain events and sends templated emails',
    'chat message relay that receives messages and broadcasts to subscribers',
    'event sourcing ledger that stores events and rebuilds state from event history',
    'retry queue manager that re-emits failed events with exponential backoff',
    'multi-service orchestrator that coordinates startup and shutdown of multiple services',
    'repository observer that watches store/update/delete and logs all changes',

    # ── Domain-Specific Applications ─────────────────────────────────────────
    'weather dashboard that fetches external API data and displays forecasts',
    'expense tracker API with categories and monthly summaries',
    'fitness log API that records workouts and computes weekly statistics',
    'plant watering scheduler that tracks schedules and emits reminder events',
    'pet feeding tracker with meal logs and portion calculations',
    'home energy monitor that reads sensor data and computes daily usage',
    'meeting room booker API with time slot availability and conflict checks',
    'quiz engine API with questions, scoring, and leaderboard',

    # ── Social & Communication ───────────────────────────────────────────────
    'microblogging platform with posts, likes, and follower feeds',
    'Mastodon-style timeline reader with live streaming updates',
    'direct message service with conversation threads and read receipts',
    'forum API with topics, replies, and moderation actions',
    'comment system API with nested replies and vote counts',

    # ── E-Commerce & Finance ─────────────────────────────────────────────────
    'shopping cart API with add, remove, and checkout workflow',
    'payment processing service with transaction state machine',
    'price comparison aggregator that fetches from multiple sources',
    'invoice generator that computes line items, tax, and totals',
    'currency converter that fetches live rates from an external API',

    # ── Templates & Output ───────────────────────────────────────────────────
    'HTML report generator using Mustache-style templates',
    'context-aware formatter that adapts output for human, machine, and developer audiences',
    'email template engine with variable substitution and conditional sections',
    'markdown-to-HTML converter that processes .md files',

    # ── Plugins & Extensions ─────────────────────────────────────────────────
    'application with a Swift plugin for custom string transformations',
    'application with a C plugin for hashing and encoding',
    'application with a Python plugin for data analysis qualifiers',
    'application using plugin qualifiers for collection operations like shuffle and sort',

    # ── Networking & Sockets ─────────────────────────────────────────────────
    'TCP echo server that accepts connections and mirrors data back',
    'socket client that connects to a remote service and exchanges messages',
    'multi-service application with HTTP server and TCP socket listener',
    'HTTP client that fetches data from an external REST API and processes the response',
    'link header pagination client that follows paginated API responses',

    # ── Date, Time & Scheduling ──────────────────────────────────────────────
    'calendar event API with recurring events and date range queries',
    'cron-style job scheduler that executes tasks at configured intervals',
    'time tracking API with start/stop timers and daily summaries',
    'date range calculator for business days excluding holidays',

    # ── Security & Auth ──────────────────────────────────────────────────────
    'API gateway with token validation and request forwarding',
    'session manager that creates, validates, and expires auth tokens',
    'access control service with roles and permission checks',

    # ── Search & Indexing ────────────────────────────────────────────────────
    'full-text search API that indexes documents and returns ranked results',
    'tag-based search API with autocomplete suggestions',
    'log search service that filters log entries by level, date, and keyword',
]

print(f'Rounds:           {NUM_ROUNDS}')
print(f'Samples / round:  {SAMPLES_PER_ROUND}')
print(f'Iters / round:    {ITERS_PER_ROUND}')
print(f'Learning rate:    {LEARNING_RATE:.0e}')
print(f'LoRA rank:        {LORA_RANK}')
print(f'LoRA layers:      {LORA_LAYERS}')
print(f'Max seq len:      {MAX_SEQ_LEN}')
print(f'Explosion guard:  >{LOSS_EXPLODE_THRESHOLD}')
print(f'Domains:          {len(DOMAINS)}')
print(f'Base model:       {BASE_MODEL}')

## Shared helpers

In [ ]:
# Load knowledge base for system prompt
kb_path = DATA_ROOT / '02_knowledge' / 'knowledge.json'
if kb_path.exists():
    with open(kb_path) as f:
        kb = json.load(f)
    action_lines = [
        f'- {"/".join(a["verbs"][:3])}  (role: {a["role"]})'
        for a in kb['actions'] if a['verbs']
    ][:35]
else:
    kb = {}
    action_lines = []
    print('WARNING: knowledge.json not found — run notebooks 01+02 first.')

ARO_SYSTEM_PROMPT = f"""You are an expert ARO language programmer.
ARO (Action Result Object) is a DSL for expressing business logic as natural-language statements.

SYNTAX:
  (FeatureSetName: BusinessActivity) {{
      Verb [the] <result[:qualifier]> preposition [the] <object[:qualifier]>.
  }}

KEY RULES:
- String concatenation: ++ (NOT +)
- Variables: hyphenated lowercase e.g. <user-id>
- Forbidden prefixes: is-, with-, empty-
- HTTP path params: Extract the <id> from the <pathParameters: id>.
- Request body: Extract the <data> from the <request: body>.
- Exactly ONE Application-Start per application
- openapi.yaml required for HTTP server
- Return: <OK: status>, <Created: status>, <NotFound: status>

AVAILABLE ACTIONS:
{chr(10).join(action_lines)}

Generate complete, valid ARO that passes `aro check`. Wrap code in ```aro fences."""


def aro_check(code):
    # tempfile.tempdir is set to DATA_ROOT/tmp in the config cell
    try:
        with tempfile.TemporaryDirectory() as tmp:
            (Path(tmp) / 'main.aro').write_text(code)
            r = subprocess.run(['aro', 'check', tmp], capture_output=True, text=True, timeout=10)
            return r.returncode == 0, (r.stderr or r.stdout).strip()[:300]
    except FileNotFoundError:
        return None, 'aro_not_found'
    except subprocess.TimeoutExpired:
        return False, 'timeout'


def extract_aro_blocks(text):
    return [b.strip() for b in re.findall(r'```aro\n(.*?)```', text, re.DOTALL) if b.strip()]


def is_comment_heavy(code):
    """Return True if >50% of non-blank lines start with (* (comment)."""
    lines = [l.strip() for l in code.splitlines() if l.strip()]
    if not lines:
        return False
    comment_lines = sum(1 for l in lines if l.startswith('(*'))
    return comment_lines / len(lines) > 0.5


def get_few_shot_examples(kb, n=3):
    """Pick n random real examples from the knowledge base for few-shot prompting."""
    examples = kb.get('examples', [])
    if not examples:
        return ''
    picked = random.sample(examples, min(n, len(examples)))
    parts = ['Here are examples of valid ARO code for reference:\n']
    for ex in picked:
        name = ex.get('name', 'Example')
        files = ex.get('aro_files', {})
        if not files:
            continue
        # Use the first (or main) .aro file
        code = list(files.values())[0][:800]
        parts.append(f'### Example: {name}\n```aro\n{code}\n```\n')
    return '\n'.join(parts) if len(parts) > 1 else ''


def make_chat_fn(model, tokenizer, temperature=0.5):
    """Return a chat function bound to the given model."""
    sampler = make_sampler(temp=temperature)
    def chat(messages, max_tokens=1200):
        prompt = tokenizer.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True
        )
        return mlx_generate(model, tokenizer, prompt=prompt, max_tokens=max_tokens,
                            sampler=sampler, verbose=False)
    return chat


def generate_with_repair(chat_fn, instruction, max_tokens=1200):
    """Self-repair loop: generate -> aro check -> fix on failure -> repeat."""
    messages = [
        {'role': 'system', 'content': ARO_SYSTEM_PROMPT},
        {'role': 'user',   'content': instruction},
    ]
    for attempt in range(MAX_REPAIR_ATTEMPTS):
        output = chat_fn(messages, max_tokens=max_tokens)
        blocks = extract_aro_blocks(output)

        if not blocks:
            messages += [
                {'role': 'assistant', 'content': output},
                {'role': 'user',      'content': 'Wrap your ARO code in ```aro\n...\n```.'},
            ]
            continue

        passed, error = aro_check('\n\n'.join(blocks))
        if passed is True or passed is None:
            return output, attempt

        messages += [
            {'role': 'assistant', 'content': output},
            {'role': 'user',      'content': f'`aro check` error:\n\n```\n{error}\n```\n\nPlease fix the code.'},
        ]
    return None, MAX_REPAIR_ATTEMPTS

## Generation: produce one round of training data

In [ ]:
def run_generation_round(round_num, model, tokenizer, n_samples):
    """Generate n_samples for this round and save to data/rounds/round_{N}.jsonl.

    Uses per-task temperature:
      - code_generation: 0.6 (creativity)
      - debugging:       0.3 (precision)
      - translation:     0.5 (balanced)

    Rejected samples (aro check failures + comment-heavy) are saved for DPO training.
    """
    output_file = ROUNDS_DIR / f'round_{round_num}.jsonl'
    reject_file = ROUNDS_DIR / f'round_{round_num}_rejects.jsonl'
    done = set()
    if output_file.exists():
        with open(output_file) as f:
            for line in f:
                try:
                    s = json.loads(line)
                    done.add(s.get('domain', s.get('source', '')))
                except Exception:
                    pass
        print(f'  Resuming round {round_num}: {len(done)} already saved')

    # Build few-shot reference from real examples
    few_shot_block = get_few_shot_examples(kb, n=3)

    stats = Counter()
    n_rejects = 0
    count = len(done)

    # Per-task temperature -> chat function mapping
    TASK_TEMPS = {
        'code_generation': 0.6,
        'debugging':       0.3,
        'translation':     0.5,
    }
    chat_fns = {
        task: make_chat_fn(model, tokenizer, temperature=temp)
        for task, temp in TASK_TEMPS.items()
    }

    with open(output_file, 'a') as out_f, open(reject_file, 'a') as rej_f:
        candidates = (DOMAINS * 20)[:n_samples * 3]
        random.shuffle(candidates)

        for domain in candidates:
            if count >= n_samples:
                break
            if domain in done:
                continue

            # 60% code generation, 30% debugging, 10% translation
            roll = random.random()
            if roll < 0.6:
                task = 'code_generation'
                instruction = (
                    f'Write a complete ARO HTTP API for: {domain}.\n'
                    f'Include openapi.yaml and main.aro with Application-Start and all operationId handlers.\n\n'
                    f'{few_shot_block}'
                    f'## openapi.yaml\n```yaml\n...\n```\n\n## main.aro\n```aro\n...\n```'
                )
            elif roll < 0.9:
                task = 'debugging'
                # Pick a random example to corrupt
                ex_codes = [list(e['aro_files'].values())[0] for e in kb.get('examples', []) if e['aro_files']]
                if not ex_codes:
                    continue
                base_code = random.choice(ex_codes)[:1500]
                bugs = [
                    'Replace one `from` with `with` in a Retrieve statement',
                    'Replace `++` with `+` for string concatenation',
                    'Remove the Application-Start feature set',
                ]
                instruction = (
                    f'Here is valid ARO code:\n\n```aro\n{base_code}\n```\n\n'
                    f'Introduce bug: {random.choice(bugs)}\n\n'
                    f'{few_shot_block}'
                    f'Then: ## Buggy Code\n```aro\n...\n```\n\n## Fix\n```aro\n...\n```'
                )
            else:
                task = 'translation'
                instruction = (
                    f'Write Python Flask pseudocode for: {domain}. '
                    f'Then translate it to ARO.\n\n'
                    f'{few_shot_block}'
                    f'## Python\n```python\n...\n```\n\n## ARO\n```aro\n...\n```'
                )

            chat_fn = chat_fns[task]
            output, attempts = generate_with_repair(chat_fn, instruction)
            stats[attempts] += 1

            if output:
                # Comment-density filter: discard if >50% of non-blank lines are comments
                blocks = extract_aro_blocks(output)
                aro_code = '\n\n'.join(blocks) if blocks else ''
                if aro_code and is_comment_heavy(aro_code):
                    # Save reject for DPO training
                    reject = {
                        'round': round_num, 'task_type': task,
                        'instruction': instruction[:200], 'output': output,
                        'domain': domain, 'reject_reason': 'comment_heavy',
                    }
                    rej_f.write(json.dumps(reject) + '\n')
                    rej_f.flush()
                    n_rejects += 1
                    continue  # don't count it, don't save it

                sample = {
                    'round': round_num, 'task_type': task,
                    'instruction': instruction[:200], 'output': output,
                    'domain': domain, 'repair_attempts': attempts,
                }
                out_f.write(json.dumps(sample) + '\n')
                out_f.flush()
                done.add(domain)
                count += 1
                if count % 20 == 0:
                    print(f'  [{count}/{n_samples}] last attempts={attempts+1}')
            else:
                # Failed aro check after all repair attempts — save reject for DPO
                reject = {
                    'round': round_num, 'task_type': task,
                    'instruction': instruction[:200], 'output': '(failed_all_repairs)',
                    'domain': domain, 'reject_reason': 'aro_check_failed',
                }
                rej_f.write(json.dumps(reject) + '\n')
                rej_f.flush()
                n_rejects += 1

    total = sum(stats.values())
    first_try_rate = 100 * stats.get(0, 0) // max(1, total)
    print(f'  Round {round_num} generation: {count} samples, {n_rejects} rejects, '
          f'first-try rate: {first_try_rate}%')
    return count

## Training: fine-tune on all rounds so far

In [ ]:
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
from IPython.display import display, clear_output

plt.style.use('default')
plt.rcParams.update({
    'text.color': '#222222',
    'axes.labelcolor': '#222222',
    'xtick.color': '#333333',
    'ytick.color': '#333333',
    'axes.titlecolor': '#111111',
    'legend.edgecolor': '#cccccc',
    'legend.facecolor': 'white',
    'legend.framealpha': 1.0,
    'figure.facecolor': '#fafafa',
    'axes.facecolor': '#f9f9f9',
})

def prepare_mlx_data(up_to_round, mlx_data_dir):
    """Combine all rounds up to up_to_round into mlx train/valid.jsonl."""
    mlx_data_dir.mkdir(parents=True, exist_ok=True)
    all_samples = []

    # Include initial dataset from notebook 10 if it exists
    initial_train = SCRIPT_DIR / '../data/05_dataset/mlx/train.jsonl'
    if initial_train.exists():
        with open(initial_train) as f:
            for line in f:
                if line.strip():
                    all_samples.append(json.loads(line))

    # Add round data
    for r in range(up_to_round + 1):
        round_file = ROUNDS_DIR / f'round_{r}.jsonl'
        if not round_file.exists():
            continue
        with open(round_file) as f:
            for line in f:
                if not line.strip():
                    continue
                s = json.loads(line)
                output = s.get('output', '')
                instruction = s.get('instruction', '')
                if not output or not instruction:
                    continue
                # Quality guard: skip junk outputs (e.g. all-punctuation, too short)
                if len(output.strip()) < 20:
                    continue
                import re as _re
                if _re.fullmatch(r'[!?.,:;\s]+', output.strip()):
                    continue
                all_samples.append({'messages': [
                    {'role': 'system',    'content': ARO_SYSTEM_PROMPT},
                    {'role': 'user',      'content': instruction},
                    {'role': 'assistant', 'content': output},
                ]})

    random.seed(42)
    random.shuffle(all_samples)
    split = int(len(all_samples) * 0.95)
    train, valid = all_samples[:split], all_samples[split:]

    def write_jsonl(data, path):
        with open(path, 'w') as f:
            for item in data:
                f.write(json.dumps(item) + '\n')

    write_jsonl(train, mlx_data_dir / 'train.jsonl')
    write_jsonl(valid, mlx_data_dir / 'valid.jsonl')
    print(f'  MLX data: {len(train)} train, {len(valid)} valid (rounds 0-{up_to_round})')
    return len(all_samples), len(train), len(valid)


def _draw_loss_plot(ax, fig, train_iters, train_losses, val_iters, val_losses, round_num):
    ax.clear()
    if train_losses:
        ax.plot(train_iters, train_losses, color='steelblue', linewidth=1.5, label='train loss')
    if val_losses:
        ax.plot(val_iters, val_losses, color='tomato', linewidth=2,
                marker='o', markersize=5, label='val loss')
    ax.set_title(f'Round {round_num} — training loss', fontsize=13)
    ax.set_xlabel('iteration')
    ax.set_ylabel('loss')
    ax.xaxis.set_major_locator(ticker.MaxNLocator(integer=True))
    ax.legend()
    ax.grid(True, alpha=0.3)
    fig.tight_layout()
    clear_output(wait=True)
    display(fig)


def run_training(round_num, data_dir, base_model_path, n_train, n_valid):
    """Run mlx_lm lora for this round, with live loss graph + stability guards."""
    adapter_dir = MODELS_DIR / f'round_{round_num}' / 'adapter'
    adapter_dir.mkdir(parents=True, exist_ok=True)
    log_path = adapter_dir / 'train.log'

    # mlx_lm requires the dataset to have at least batch_size examples.
    effective_batch = max(1, min(BATCH_SIZE, n_train, n_valid))
    val_batches     = max(1, min(10, n_valid // effective_batch))

    cmd = [
        sys.executable, '-m', 'mlx_lm', 'lora',
        '--model',               str(base_model_path),
        '--data',                str(data_dir),
        '--train',
        '--batch-size',          str(effective_batch),
        '--lora-rank',           str(LORA_RANK),
        '--num-layers',          str(LORA_LAYERS),
        '--learning-rate',       str(LEARNING_RATE),
        '--iters',               str(ITERS_PER_ROUND),
        '--save-every',          str(ITERS_PER_ROUND // 2),
        '--val-batches',         str(val_batches),
        '--adapter-path',        str(adapter_dir),
        '--max-seq-length',      str(MAX_SEQ_LEN),
        '--mask-prompt',
    ]

    print(f'  Training round {round_num} '
          f'(batch={effective_batch}, val_batches={val_batches}, lr={LEARNING_RATE:.0e}, '
          f'rank={LORA_RANK}, layers={LORA_LAYERS}, max_seq={MAX_SEQ_LEN})...')
    print(f'  Log: {log_path}')

    re_train = re.compile(r'Iter\s+(\d+):\s+Train loss\s+([\d.]+)', re.IGNORECASE)
    re_val   = re.compile(r'Iter\s+(\d+):\s+Val loss\s+([\d.]+)',   re.IGNORECASE)

    train_iters, train_losses = [], []
    val_iters,   val_losses   = [], []
    all_lines = []

    # Stability tracking
    _val_loss_streak = 0
    _prev_val_loss   = None
    _stopped_early   = False
    _loss_exploded   = False

    fig, ax = plt.subplots(figsize=(8, 4))

    proc = subprocess.Popen(
        cmd,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )

    update_every = 5
    last_draw    = 0

    with open(log_path, 'w') as log_f:
        for line in proc.stdout:
            line = line.rstrip()
            all_lines.append(line)
            log_f.write(line + '\n')
            log_f.flush()

            m = re_train.search(line)
            if m:
                it   = int(m.group(1))
                loss = float(m.group(2))
                train_iters.append(it)
                train_losses.append(loss)

                # Loss explosion guard
                if loss > LOSS_EXPLODE_THRESHOLD:
                    _loss_exploded = True
                    print(f'\n  LOSS EXPLOSION at iter {it}: train_loss={loss:.3f} > {LOSS_EXPLODE_THRESHOLD}')
                    print('    Killing training. Reduce LEARNING_RATE and retry.')
                    proc.terminate()
                    break

                if len(train_iters) - last_draw >= update_every:
                    _draw_loss_plot(ax, fig, train_iters, train_losses,
                                    val_iters, val_losses, round_num)
                    last_draw = len(train_iters)
                continue

            m = re_val.search(line)
            if m:
                it   = int(m.group(1))
                loss = float(m.group(2))
                val_iters.append(it)
                val_losses.append(loss)
                _draw_loss_plot(ax, fig, train_iters, train_losses,
                                val_iters, val_losses, round_num)
                last_draw = len(train_iters)

                # Early stopping check
                if _prev_val_loss is not None and loss > _prev_val_loss:
                    _val_loss_streak += 1
                    print(f'  val_loss up ({_prev_val_loss:.4f}->{loss:.4f}), '
                          f'streak {_val_loss_streak}/{EARLY_STOP_PATIENCE}')
                    if _val_loss_streak >= EARLY_STOP_PATIENCE:
                        _stopped_early = True
                        print(f'\n  EARLY STOP round {round_num} at iter {it} — val_loss diverging.')
                        proc.terminate()
                        break
                else:
                    _val_loss_streak = 0
                _prev_val_loss = loss

    proc.wait()

    # Final plot
    if train_losses or val_losses:
        _draw_loss_plot(ax, fig, train_iters, train_losses,
                        val_iters, val_losses, round_num)
    plt.close(fig)

    if _loss_exploded:
        raise RuntimeError(
            f'Round {round_num} aborted: loss explosion (>{LOSS_EXPLODE_THRESHOLD}). '
            f'Reduce LEARNING_RATE to {LEARNING_RATE/2:.0e}.'
        )
    if _stopped_early:
        best_it = val_iters[val_losses.index(min(val_losses))] if val_losses else '?'
        print(f'  Early stopped. Best val checkpoint at iter {best_it}.')
    elif proc.returncode not in (0, -15):
        tail = all_lines[-40:] if len(all_lines) >= 40 else all_lines
        print(f'\n--- training failed (last {len(tail)} lines) ---')
        print('\n'.join(tail))
        print(f'--- full log: {log_path} ---')
        raise RuntimeError(f'Training failed at round {round_num} (exit {proc.returncode})')

    # Log summary
    if train_losses:
        print(f'  Train loss: {train_losses[0]:.4f} -> {train_losses[-1]:.4f}  '
              f'(best {min(train_losses):.4f})')
    if val_losses:
        print(f'  Val   loss: {val_losses[0]:.4f} -> {val_losses[-1]:.4f}  '
              f'(best {min(val_losses):.4f})')

    return adapter_dir


def fuse_model(round_num, base_model_path, adapter_dir):
    """Merge adapter into base weights for the next round's generation."""
    fused_dir = MODELS_DIR / f'round_{round_num}' / 'fused'
    cmd = [
        sys.executable, '-m', 'mlx_lm', 'fuse',
        '--model',        str(base_model_path),
        '--adapter-path', str(adapter_dir),
        '--save-path',    str(fused_dir),
    ]
    print(f'  Fusing round {round_num}...')
    result = subprocess.run(cmd)
    if result.returncode != 0:
        raise RuntimeError(f'Fuse failed at round {round_num}')
    return fused_dir

## Evaluation helper (syntax pass rate)

In [7]:
def quick_eval(model, tokenizer, n=30):
    """Generate n code samples and measure aro check pass rate."""
    chat_fn = make_chat_fn(model, tokenizer)
    passed = 0
    for domain in random.sample(DOMAINS, min(n, len(DOMAINS))):
        instruction = f'Write a short ARO feature set that handles {domain}.'
        messages = [
            {'role': 'system', 'content': ARO_SYSTEM_PROMPT},
            {'role': 'user',   'content': instruction},
        ]
        prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        output = mlx_generate(model, tokenizer, prompt=prompt, max_tokens=600, verbose=False)
        blocks = extract_aro_blocks(output)
        if blocks:
            ok, _ = aro_check('\n\n'.join(blocks))
            if ok is True:
                passed += 1
    rate = passed / n
    print(f'  Syntax pass rate: {passed}/{n} = {rate:.0%}')
    return rate

## Run the loop

In [ ]:
round_metrics = []
current_model_path = BASE_MODEL   # starts as MODEL_ID (via config.py), becomes local path after round 0

# Evaluate base model before any training
print('=== Baseline evaluation ===')
print(f'Loading {current_model_path} ...')
model, tokenizer = load(str(current_model_path))
baseline_rate = quick_eval(model, tokenizer)
round_metrics.append({'round': -1, 'model': str(current_model_path), 'syntax_pass_rate': baseline_rate})
del model   # free VRAM between generation and training

In [ ]:
for round_num in range(NUM_ROUNDS):
    t_start = time.time()
    print(f'\n{"="*60}')
    print(f'ROUND {round_num}')
    print(f'{"="*60}')

    # --- 1. Load current model for generation ---
    print(f'\n[1/6] Loading model: {current_model_path}')
    model, tokenizer = load(str(current_model_path))

    # --- 2. Generate (per-task temperature handled inside) ---
    print(f'\n[2/6] Generating {SAMPLES_PER_ROUND} samples...')
    t_gen = time.time()
    n_generated = run_generation_round(round_num, model, tokenizer, SAMPLES_PER_ROUND)
    gen_min = (time.time() - t_gen) / 60
    print(f'  Generation done: {n_generated} samples in {gen_min:.1f} min '
          f'({n_generated / max(gen_min, 0.1):.0f} samples/min)')

    # Free model memory before training
    del model, tokenizer

    # --- 3. Prepare combined MLX dataset ---
    print(f'\n[3/6] Preparing dataset (rounds 0-{round_num})...')
    mlx_data_dir = ROUNDS_DIR / f'mlx_round_{round_num}'
    total_samples, n_train, n_valid = prepare_mlx_data(round_num, mlx_data_dir)
    print(f'  Cumulative dataset: {total_samples:,} total ({n_train:,} train / {n_valid:,} valid)')

    # --- 4. Fine-tune ---
    print(f'\n[4/6] Fine-tuning (iters={ITERS_PER_ROUND}, lr={LEARNING_RATE:.0e}, '
          f'rank={LORA_RANK}, layers={LORA_LAYERS}, max_seq={MAX_SEQ_LEN})...')
    t_train = time.time()
    adapter_dir = run_training(round_num, mlx_data_dir, current_model_path, n_train, n_valid)
    train_min = (time.time() - t_train) / 60
    print(f'  Training done in {train_min:.1f} min')

    # --- 5. Fuse adapter -> new model ---
    print(f'\n[5/6] Fusing adapter into base model...')
    t_fuse = time.time()
    fused_dir = fuse_model(round_num, current_model_path, adapter_dir)
    fuse_sec = time.time() - t_fuse
    current_model_path = fused_dir
    print(f'  Fused in {fuse_sec:.0f}s -> {fused_dir}')

    # --- 6. Evaluate new model ---
    print(f'\n[6/6] Evaluating round {round_num} model...')
    model, tokenizer = load(str(fused_dir))
    pass_rate = quick_eval(model, tokenizer)
    del model

    elapsed = (time.time() - t_start) / 60
    prev_rate = round_metrics[-1]['syntax_pass_rate'] if round_metrics else 0
    delta = pass_rate - prev_rate

    round_metrics.append({
        'round':            round_num,
        'model':            str(fused_dir),
        'n_generated':      n_generated,
        'total_dataset':    total_samples,
        'syntax_pass_rate': pass_rate,
        'minutes':          round(elapsed, 1),
        'gen_minutes':      round(gen_min, 1),
        'train_minutes':    round(train_min, 1),
    })

    # Round summary
    print(f'\n{"~"*60}')
    print(f'  Round {round_num} complete in {elapsed:.0f} min')
    print(f'    Generated:    {n_generated} samples ({gen_min:.1f} min)')
    print(f'    Dataset:      {total_samples:,} cumulative')
    print(f'    Training:     {train_min:.1f} min')
    print(f'    Pass rate:    {pass_rate:.0%}  ({"+" if delta >= 0 else ""}{delta:.0%} vs previous)')
    print(f'{"~"*60}')

    # Save progress
    with open(MODELS_DIR / 'loop_metrics.json', 'w') as f:
        json.dump(round_metrics, f, indent=2)

## Results

In [10]:
print('\n=== Self-Improvement Summary ===\n')
print(f'{"Round":<8} {"Pass Rate":<12} {"Dataset":<12} {"Minutes":<10} {"Model"}')
print('─' * 70)
for m in round_metrics:
    r     = m['round']
    label = 'baseline' if r == -1 else str(r)
    rate  = f'{m["syntax_pass_rate"]:.0%}'
    ds    = str(m.get('total_dataset', '—'))
    mins  = str(m.get('minutes', '—'))
    mod   = Path(m['model']).name if Path(m['model']).exists() else m['model'].split('/')[-1]
    print(f'{label:<8} {rate:<12} {ds:<12} {mins:<10} {mod}')

print(f'\nFinal model: {current_model_path}')
print(f'Metrics:     {ITERATIVE_MODELS_DIR / "loop_metrics.json"}')


=== Self-Improvement Summary ===

Round    Pass Rate    Dataset      Minutes    Model
──────────────────────────────────────────────────────────────────────
baseline 3%           —            —          fused
0        33%          922          96.1       fused
1        30%          1044         70.3       fused
2        43%          1153         124.6      fused

Final model: /Users/kris/Projects/ARO/ARO-Train/Train/models/round_2/fused
Metrics:     /Users/kris/Projects/ARO/ARO-Train/Train/models/iterative/loop_metrics.json


In [11]:
# ── Self-improvement graph ──────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

rounds  = [m['round'] for m in round_metrics]
labels  = ['base' if r == -1 else f'R{r}' for r in rounds]
rates   = [m['syntax_pass_rate'] for m in round_metrics]

# --- Panel 1: Syntax pass rate ---
ax = axes[0]
colors = ['#888888' if r == -1 else 'steelblue' for r in rounds]
bars = ax.bar(labels, [r * 100 for r in rates], color=colors, edgecolor='white', width=0.6)
for bar, rate in zip(bars, rates):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 1,
            f'{rate:.0%}', ha='center', va='bottom', fontsize=11, fontweight='bold')
ax.set_ylabel('pass rate (%)')
ax.set_title('Syntax pass rate per round', fontsize=13)
ax.set_ylim(0, max(r * 100 for r in rates) * 1.25 + 5)
ax.grid(axis='y', alpha=0.3)

# --- Panel 2: Cumulative dataset size ---
ax = axes[1]
trained = [m for m in round_metrics if m['round'] >= 0]
if trained:
    t_labels = [f'R{m["round"]}' for m in trained]
    datasets = [m.get('total_dataset', 0) for m in trained]
    ax.bar(t_labels, datasets, color='mediumseagreen', edgecolor='white', width=0.6)
    for i, (lbl, ds) in enumerate(zip(t_labels, datasets)):
        ax.text(i, ds + max(datasets) * 0.02, f'{ds:,}', ha='center', va='bottom', fontsize=10)
    ax.set_ylabel('samples')
    ax.set_title('Cumulative dataset size', fontsize=13)
    ax.grid(axis='y', alpha=0.3)
else:
    ax.text(0.5, 0.5, 'no training rounds', ha='center', va='center', transform=ax.transAxes)
    ax.set_title('Cumulative dataset size', fontsize=13)

# --- Panel 3: Time breakdown ---
ax = axes[2]
if trained:
    gen_mins   = [m.get('gen_minutes', 0) for m in trained]
    train_mins = [m.get('train_minutes', 0) for m in trained]
    x = range(len(trained))
    ax.bar(t_labels, gen_mins, color='sandybrown', edgecolor='white', width=0.6, label='generation')
    ax.bar(t_labels, train_mins, bottom=gen_mins, color='cornflowerblue',
           edgecolor='white', width=0.6, label='training')
    ax.set_ylabel('minutes')
    ax.set_title('Time per round', fontsize=13)
    ax.legend(fontsize=9)
    ax.grid(axis='y', alpha=0.3)
else:
    ax.text(0.5, 0.5, 'no training rounds', ha='center', va='center', transform=ax.transAxes)
    ax.set_title('Time per round', fontsize=13)

fig.suptitle('NB17 — Iterative Self-Improvement', fontsize=15, fontweight='bold', y=1.02)
fig.tight_layout()
plt.savefig(MODELS_DIR / 'loop_summary.png', dpi=150, bbox_inches='tight', facecolor='#fafafa')
plt.show()
print(f'Saved: {MODELS_DIR / "loop_summary.png"}')

Saved: /Users/kris/Projects/ARO/ARO-Train/Train/models/loop_summary.png


## Summary

In [12]:
print('=' * 65)
print('NOTEBOOK 17 — ITERATIVE SELF-IMPROVEMENT SUMMARY')
print('=' * 65)

print(f'\nConfiguration:')
print(f'  Rounds:          {NUM_ROUNDS}')
print(f'  Samples/round:   {SAMPLES_PER_ROUND}')
print(f'  Iters/round:     {ITERS_PER_ROUND}')
print(f'  Learning rate:   {LEARNING_RATE:.0e}')
print(f'  Batch size:      {BATCH_SIZE}')
print(f'  LoRA layers:     {LORA_LAYERS}')

if round_metrics:
    baseline = next((m for m in round_metrics if m['round'] == -1), None)
    final    = round_metrics[-1]

    print(f'\nProgress:')
    for m in round_metrics:
        r     = m['round']
        label = 'baseline' if r == -1 else f'round {r}'
        rate  = f'{m["syntax_pass_rate"]:.0%}'
        ds    = f'{m.get("total_dataset","?"):,}' if isinstance(m.get("total_dataset"), int) else '—'
        mins  = f'{m.get("minutes","?"):.0f}m' if isinstance(m.get("minutes"), (int, float)) else '—'
        print(f'  {label:<12}  pass_rate={rate:<6}  dataset={ds:<8}  time={mins}')

    if baseline and len(round_metrics) > 1:
        start_rate = baseline['syntax_pass_rate']
        end_rate   = final['syntax_pass_rate']
        delta      = end_rate - start_rate
        print(f'\nImprovement: {start_rate:.0%} → {end_rate:.0%}  (Δ {delta:+.0%})')
        if end_rate < 0.10:
            print('  ⚠  Syntax pass rate still very low (<10%). Model is not learning ARO.')
            print('     Check data quality in NB14 and ensure aro check works on generated samples.')
        elif end_rate < 0.30:
            print('  ⚠  Modest improvement but still low. Run more rounds or increase SAMPLES_PER_ROUND.')
        else:
            print('  ✓  Meaningful syntax pass rate improvement.')
else:
    print('\n  (no round metrics recorded)')

print(f'\nFinal model: {current_model_path}')
print(f'Metrics log: {MODELS_DIR / "loop_metrics.json"}')

print(f'\nNext steps:')
print(f'  1. Run NB16 (evaluation) on final model to get full metric breakdown.')
print(f'  2. If pass_rate < 20%, re-run NB14 to rebuild dataset with better balance.')
print(f'  3. If loss exploded in any round, LEARNING_RATE is still too high — try {LEARNING_RATE/2:.0e}.')
print('=' * 65)

NOTEBOOK 17 — ITERATIVE SELF-IMPROVEMENT SUMMARY

Configuration:
  Rounds:          3
  Samples/round:   300
  Iters/round:     400
  Learning rate:   1e-04
  Batch size:      4
  LoRA layers:     8

Progress:
  baseline      pass_rate=3%      dataset=—         time=—
  round 0       pass_rate=33%     dataset=922       time=96m
  round 1       pass_rate=30%     dataset=1,044     time=70m
  round 2       pass_rate=43%     dataset=1,153     time=125m

Improvement: 3% → 43%  (Δ +40%)
  ✓  Meaningful syntax pass rate improvement.

Final model: /Users/kris/Projects/ARO/ARO-Train/Train/models/round_2/fused
Metrics log: /Users/kris/Projects/ARO/ARO-Train/Train/models/loop_metrics.json

Next steps:
  1. Run NB16 (evaluation) on final model to get full metric breakdown.
  2. If pass_rate < 20%, re-run NB14 to rebuild dataset with better balance.
  3. If loss exploded in any round, LEARNING_RATE is still too high — try 5e-05.
